In [0]:

#cdc_col(chnage data capture) like modified date when data is updated or changed
cdc_column = 'modified_date'
source_schema = 'silver'

source_object = 'silver_bookings'

fact_table = f"workspace.{source_schema}.{source_object}"
target_schema = 'gold'

target_object = 'factBookings'

backdated_refresh = ''

#fact key columns list

fact_key_cols =  ['DimPassengers_id_key','DimFlights_id_key','DimAirports_id_key','booking_date']




In [0]:
from pyspark.sql.functions import * 

In [0]:
dimensions = [
    {
        "table" : "workspace.gold.DimPassengers",
        "alias" : "DimPassengers",
        "join_keys" : [('passenger_id','passenger_id')] #(fact_col,dim_col)
    },
    {
        "table" : "workspace.gold.DimFlights",
        "alias" : "DimFlights",
        "join_keys" : [('flight_id','flight_id')]
    },
     {
        "table" : "workspace.gold.DimAirports",
        "alias" : "DimAirports",
        "join_keys" : [('airport_id','airport_id')]
    }
]
#columns you want to keep in the fact table beside dimension surrogate keys
fact_columns  = ['amount','booking_date','modified_date']

In [0]:
if len(backdated_refresh) ==  0:
    #if table exists in destination
    if spark.catalog.tableExists(f"workspace.{target_schema}.{target_object}"):
        last_load = spark.sql(f"select max({cdc_column}) from {target_schema}.{target_object}").collect()[0][0]
    
    else:
        last_load = '1900-01-01 00:00:00'
else:
    last_load = backdated_refresh


last_load
    

### Dynamic fact query(bring keys)

In [0]:
import ast

def generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_col, processing_date):
    fact_alias = 'f'

    select_cols = [f"{fact_alias}.{col}" for col in fact_columns]

    join_clauses = []
    for dim in dimensions:
        table_full = dim['table']
        alias = dim['alias']
        table_name = table_full.split('.')[-1]
        surrogate_key = f"{alias}.{table_name}_id_key"
        select_cols.append(surrogate_key)

        # Build ON clause (per dimension)
        on_conditions = [
            f"{fact_alias}.{fk} = {alias}.{dk}" for fk, dk in dim['join_keys']
        ]
        join_clause = f"LEFT JOIN {table_full} {alias} ON " + " AND ".join(on_conditions)
        join_clauses.append(join_clause)

    # Final SELECT and JOIN clauses
    select_clause = ",\n    ".join(select_cols)
    joins = "\n".join(join_clauses)

    # WHERE clause for incremental filtering
    where_clause = f"{fact_alias}.{cdc_column} >= DATE('{processing_date}')"

    # Final query
    query = f"""
SELECT
    {select_clause}
FROM {fact_table} {fact_alias}
{joins}
WHERE {where_clause}
""".strip()

    return query


# ---- Inputs ----

fact_table = "workspace.silver.silver_bookings"

# fact_columns as a real list, not a stringified tuple
fact_columns_str = "['amount', 'booking_date', 'modified_date']"
fact_columns = ast.literal_eval(fact_columns_str)



# ---- Generate query ----

query = generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_column, last_load)
print(query)

In [0]:
df_fact = spark.sql(query)

In [0]:
df_fact.display()

### UPSERT

In [0]:
fact_key_cols_str = " AND ".join([f"srce.{col} = trget.{col}" for col in fact_key_cols])


In [0]:
from delta.tables import DeltaTable


if spark.catalog.tableExists(f"workspace.{target_schema}.{target_object}"):
    dlt_obj = DeltaTable.forName(spark,f"workspace.{target_schema}.{target_object}")
    dlt_obj.alias("trget").merge(df_fact.alias("srce"),fact_key_cols_str)\
                            .whenMatchedUpdateAll(condition = f"srce.{cdc_column} = trget.{cdc_column}")\
                            .whenNotMatchedInsertAll()\
                            .execute()
else:
    df_fact.write.format("delta")\
                .mode("append")\
                .saveAsTable(f"workspace.{target_schema}.{target_object}")

In [0]:
df = spark.sql('select * from gold.dimpassengers').groupBy("DimPassengers_id_key").count().filter(col('count')>1)
df.display()